# Entrenamiento Completo: Transformer, Baseline y Ablación

En este notebook realizaremos los tres experimentos académicos requeridos:
1. **MasterStockTransformer (Principal):** Modelo de Atención Cruzada con todo el contexto.
2. **LSTM Baseline:** Modelo clásico LSTM para comparar rendimientos.
3. **Estudio de Ablación (Ciego al Mercado):** Transformer entrenado *únicamente* con datos de JPM (sin mirar el XLF ni a los otros bancos) para probar que agregar el contexto macroeconómico mejora las predicciones.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from model import StockDataset, MasterStockTransformer, LSTMModel

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

Dispositivo de entrenamiento: mps


### 1. Preparación de Datos Base

In [2]:
# Cargar datos
df = pd.read_csv('processed_market_data.csv', index_col=0, parse_dates=True)

target_col_name = 'JPM_Log_Ret'
target_idx = df.columns.get_loc(target_col_name)

ma5_idx = df.columns.get_loc('JPM_Target_MA5')
close_idx = df.columns.get_loc('JPM_Close')

# Contexto Normal (Todo el mercado)
context_indices = [i for i in range(len(df.columns)) if i not in [target_idx, ma5_idx, close_idx]]

# Contexto Ablación (Solo JPM y Tiempo)
jpm_and_time_cols = [c for c in df.columns if ('JPM' in c or 'Sin' in c or 'Cos' in c) and c not in [target_col_name, 'JPM_Target_MA5', 'JPM_Close']]
context_indices_ablation = [df.columns.get_loc(c) for c in jpm_and_time_cols]

# Data Split
train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:train_size+val_size]
test_df = df.iloc[train_size+val_size:]

# Escalamiento
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

# DataLoaders
seq_length = 20
batch_size = 64

def create_loaders(ctx_indices):
    train_set = StockDataset(train_scaled, target_idx, ctx_indices, seq_length)
    val_set = StockDataset(val_scaled, target_idx, ctx_indices, seq_length)
    t_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    v_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    return t_loader, v_loader

train_loader_full, val_loader_full = create_loaders(context_indices)
train_loader_abl, val_loader_abl = create_loaders(context_indices_ablation)

### 2. Función de Entrenamiento Universal

In [3]:
def train_model(model, train_loader, val_loader, filename, epochs=50, lr=0.0005):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0
        for x_tgt, x_ctx, y_true in train_loader:
            x_tgt, x_ctx, y_true = x_tgt.to(device), x_ctx.to(device), y_true.to(device)
            optimizer.zero_grad()
            predictions, _ = model(x_tgt, x_ctx)
            loss = criterion(predictions, y_true)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_train_loss += loss.item()
            
        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for x_tgt, x_ctx, y_true in val_loader:
                x_tgt, x_ctx, y_true = x_tgt.to(device), x_ctx.to(device), y_true.to(device)
                preds, _ = model(x_tgt, x_ctx)
                loss = criterion(preds, y_true)
                epoch_val_loss += loss.item()
                
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train MSE: {epoch_train_loss/len(train_loader):.5f} | Val MSE: {epoch_val_loss/len(val_loader):.5f}")
            
    torch.save(model.state_dict(), filename)
    print(f"-> ¡Modelo guardado en {filename}!\n")

### 3. Ejecución de los 3 Experimentos (Con los Mejores Hiperparámetros)

In [4]:
# HIPERPARÁMETROS ÓPTIMOS DEL GRID SEARCH
OPTIMAL_EPOCHS = 30
OPTIMAL_LAYERS = 1
OPTIMAL_DROPOUT = 0.1

print("=== EXPERIMENTO 1: MASTER Stock Transformer (Mercado Completo) ===")
master_model = MasterStockTransformer(target_features=1, context_features=len(context_indices), 
                                      d_model=128, nhead=4, num_layers=OPTIMAL_LAYERS, dropout=OPTIMAL_DROPOUT).to(device)
train_model(master_model, train_loader_full, val_loader_full, 'master_model_tuned.pth', epochs=OPTIMAL_EPOCHS)

print("=== EXPERIMENTO 2: LSTM Baseline (Mercado Completo) ===")
# Aplicamos 1 capa y dropout 0 (porque LSTM se queja si hay 1 capa y dropout > 0)
lstm_model = LSTMModel(target_features=1, context_features=len(context_indices), 
                       hidden_size=128, num_layers=OPTIMAL_LAYERS, dropout=0.0).to(device)
train_model(lstm_model, train_loader_full, val_loader_full, 'baseline_lstm.pth', epochs=OPTIMAL_EPOCHS)

print("=== EXPERIMENTO 3: Estudio de Ablación (Ciego / Solo JPM) ===")
ablation_model = MasterStockTransformer(target_features=1, context_features=len(context_indices_ablation), 
                                        d_model=128, nhead=4, num_layers=OPTIMAL_LAYERS, dropout=OPTIMAL_DROPOUT).to(device)
train_model(ablation_model, train_loader_abl, val_loader_abl, 'master_model_ablation.pth', epochs=OPTIMAL_EPOCHS)

=== EXPERIMENTO 1: MASTER Stock Transformer (Mercado Completo) ===
Epoch 1/30 | Train MSE: 1.04474 | Val MSE: 0.73411
Epoch 10/30 | Train MSE: 0.97889 | Val MSE: 0.75806
Epoch 20/30 | Train MSE: 0.89167 | Val MSE: 0.73713
Epoch 30/30 | Train MSE: 0.84801 | Val MSE: 0.73750
-> ¡Modelo guardado en master_model_tuned.pth!

=== EXPERIMENTO 2: LSTM Baseline (Mercado Completo) ===
Epoch 1/30 | Train MSE: 1.00465 | Val MSE: 0.71343
Epoch 10/30 | Train MSE: 0.88692 | Val MSE: 0.74967
Epoch 20/30 | Train MSE: 0.72729 | Val MSE: 0.85492
Epoch 30/30 | Train MSE: 0.55541 | Val MSE: 1.14116
-> ¡Modelo guardado en baseline_lstm.pth!

=== EXPERIMENTO 3: Estudio de Ablación (Ciego / Solo JPM) ===
Epoch 1/30 | Train MSE: 1.06716 | Val MSE: 0.73116
Epoch 10/30 | Train MSE: 0.97217 | Val MSE: 0.74374
Epoch 20/30 | Train MSE: 0.90967 | Val MSE: 0.71804
Epoch 30/30 | Train MSE: 0.93384 | Val MSE: 0.73033
-> ¡Modelo guardado en master_model_ablation.pth!

